![Banner](../../../../Images/Banner/07_remote_sensing/09_03_banner_rs_rader.png)


# NISAR Processing

This notebook describes how to process NISAR products, from reading the HDF5 files to the steps that are specific to each Level-1 product in radar geometry and each Level-2 geocoded product.


## Overview

NISAR differs from Sentinel-1 in two ways that change the processing: the products come in **HDF5** rather than SAFE/GeoTIFF, and most science products are **already geocoded (Level-2)**. Many steps that take a long SNAP chain for Sentinel-1 (calibration, terrain correction, coregistration) are already done. The work shifts toward reading HDF5 correctly, masking, quality control, time series analysis, and corrections such as ionosphere and troposphere.

The HDF5 paths below follow the NISAR product specifications. Check them against your files with `h5py` (`f.visit(print)`), since early releases have changed some names.


## Core concepts

### Learning objectives

1. Choose the NISAR product for the goal: RSLC, RIFG, RUNW, ROFF, GSLC, GCOV, GUNW, or GOFF.
2. Read HDF5 identification metadata and list the frequencyA swath or grid before loading large data layers.
3. Follow the steps common to every product: Earthdata access, search, release checks, and a fixed track, frame, frequency, and polarization.
4. Describe Level-1 radar-geometry processing for RSLC, wrapped RIFG, unwrapped RUNW, and ROFF offsets, including geolocation-grid geocoding.
5. Describe Level-2 map-grid processing for GSLC interferograms, GCOV covariance (gamma0), GUNW unwrapped phase, and GOFF offsets.
6. Use the product summary to match each product to its geometry, main processing steps, and typical output.
7. Name open-source tools used for NISAR and InSAR processing, including ISCE3, SNAP, GMTSAR, and MintPy.

### Contents

1. Overview
2. Planning and data selection
3. Reading NISAR HDF5 files (common to all workflows)
4. NISAR Data Products: Processing Steps
5. Steps common to all products
6. Level 1 products (radar geometry)
7. RSLC: Range-Doppler Single Look Complex
8. RIFG: Wrapped interferogram (radar geometry)
9. RUNW: Unwrapped interferogram (radar geometry)
10. ROFF: Pixel offsets (radar geometry)
11. Geocoding Level 1 products
12. Level 2 products (geocoded)
13. GSLC: Geocoded Single Look Complex
14. GCOV: Geocoded polarimetric covariance (γ⁰)
15. GUNW: Geocoded unwrapped interferogram
16. GOFF: Geocoded pixel offsets
17. Summary: which product and what to do
18. NISAR Processing with GeoLattice
19. Summary, exercises and references


## Planning and data selection

1. **Choose the product for the goal**

| Goal | Product |
|---|---|
| Backscatter, land cover, crops, biomass, flooding | GCOV (L2) |
| Your own interferograms, coherence | GSLC (L2) |
| Deformation time series | GUNW (L2) |
| Glacier and ice velocity, large earthquake offsets | GOFF (L2) |
| Custom processing from scratch | RSLC (L1) with ISCE3 |
| Soil moisture, ecosystem, displacement maps | L3 products |

2. **Keep geometry consistent:** same track, same frame, same orbit direction (ascending or descending)
3. **Keep radar settings consistent:** same frequency band (A or B), same polarization, and same bandwidth. Interferograms across different bandwidth modes need extra resampling
4. **Know the frequency sub-bands**
   - `frequencyA`: main band (higher bandwidth, main science)
   - `frequencyB`: narrower auxiliary band in some modes, used for split-spectrum ionosphere correction
5. **Check polarization mode:** single (HH), dual (HH/HV), quad (HH/HV/VH/VV), or compact pol
6. **Read the known-issues list** in the ASF NISAR Data User Guide for the release you use
7. **Decide where to process:** downloading works for small areas. For large stacks, work in NASA Earthdata Cloud (AWS us-west-2) and stream from S3
8. **Set up credentials:** NASA Earthdata login (`~/.netrc` or `earthaccess.login()`)


## Reading NISAR HDF5 files (common to all workflows)

### Typical file structure

```
/science/LSAR/
    identification/          # mission, track, frame, orbit direction, dates
    GCOV/ (or GSLC, GUNW, GOFF)
        grids/
            frequencyA/
                HHHH, HVHV, ...      # data layers
                xCoordinates, yCoordinates
                projection           # EPSG code as attribute
                mask, numberOfLooks, ...
        metadata/
            radarGrid/               # incidence angle, LOS vectors, slant range (3D cubes)
            calibrationInformation/
            processingInformation/
            orbit/, attitude/
```

### Steps

1. Open the file with `h5py` (or `xarray` with the `h5netcdf` engine)
2. List the structure and confirm layer names
3. Read the EPSG code from the `projection` dataset attributes
4. Read `xCoordinates` and `yCoordinates` to build the georeference (pixel centers)
5. Read the data layer and the mask
6. Set fill values (often NaN or 0) to missing
7. Wrap the result in an `xarray.DataArray` with `rioxarray` CRS and transform
8. Clip to your AOI and save as Cloud Optimized GeoTIFF or Zarr


## NISAR Data Products: Processing Steps

The eight products form one processing chain. Level 1 stays in **radar geometry** (slant range × azimuth time), and Level 2 is the same information **geocoded** to a map grid.

```
                RSLC (L1, complex, radar geometry)
               /          |             \
          RIFG          ROFF          GSLC (L2) ──> GCOV (L2)
       (wrapped)     (offsets)          |
           |             |           (your own
          RUNW         GOFF (L2)    interferograms)
       (unwrapped)
           |
          GUNW (L2)
```

**Rule of thumb:** use Level 2 unless you need custom geocoding, a custom grid, your own unwrapping, or full radar-geometry control. Level 1 gives more control but needs geocoding at the end.

The HDF5 paths below follow the NISAR product specifications. Confirm them in your files with `h5py` (`f.visit(print)`), since names and units have shifted between releases.


## Steps common to all products

1. Log in to NASA Earthdata (`earthaccess.login()` or `~/.netrc`)
2. Search with `asf_search` or `earthaccess` by AOI, dates, product type, track, frame, and orbit direction
3. Check the release version (Beta, Provisional, Validated) and its known issues
4. Keep track, frame, orbit direction, frequency (A or B), polarization, and bandwidth the same across pairs or stacks
5. Open the HDF5 and list its structure
6. Read the `identification` group: mission, track, frame, orbit direction, start and end times, look direction
7. Read data layers along with their units and fill values from dataset attributes
8. Do the product-specific steps below
9. Save outputs as Cloud Optimized GeoTIFF, Zarr, or NetCDF, with CRS and metadata


## Level 1 products (radar geometry)

Level 1 files keep data on a `swaths` grid indexed by `zeroDopplerTime` (azimuth) and `slantRange`. Geolocation comes from a **geolocation grid** in the metadata: 3D cubes of map coordinates (x, y) at several heights, plus incidence angle and LOS vectors. You need it, or ISCE3, to put Level 1 results on a map.

## RSLC: Range-Doppler Single Look Complex

**What it is:** the focused, calibrated complex radar image (amplitude and phase) in slant range and zero-Doppler azimuth. It is the base for every other product.

**Key layers**
- `/science/LSAR/RSLC/swaths/frequencyA/HH` (and HV, VH, VV): complex64
- `.../frequencyA/slantRange`, `.../zeroDopplerTime`
- `/science/LSAR/RSLC/metadata/geolocationGrid/`: coordinateX, coordinateY, incidenceAngle, losUnitVectorX/Y
- `metadata/orbit`, `metadata/attitude`, `metadata/calibrationInformation`

**When to use:** custom InSAR pairs, custom polarimetry, your own GSLC or GCOV with a chosen grid and DEM, research on focusing or calibration.

### Path A: Amplitude and backscatter in radar geometry
1. Read the complex layer for each polarization
2. Compute power: |s|² (RSLC is calibrated to β⁰)
3. Convert β⁰ to σ⁰ with the incidence angle: σ⁰ = β⁰ × sin(θ). For hilly terrain, use radiometric terrain correction (RTC) for γ⁰ instead (done by ISCE3 GCOV workflow)
4. Multilook to roughly square pixels (for example 2 range × 8 azimuth, depending on mode)
5. Apply a speckle filter
6. Geocode (see "Geocoding Level 1 products" below)
7. Convert to dB

### Path B: Polarimetry in radar geometry (quad-pol)
1. Read HH, HV, VH, VV complex layers
2. Symmetrize: HV_sym = (HV + VH) / 2
3. Build the scattering vector and C3 or T3 matrix per pixel
4. Multilook (average the matrix over a window)
5. Polarimetric speckle filter (Refined Lee)
6. Decompose (Freeman-Durden, Yamaguchi, H-A-α)
7. Geocode the outputs

### Path C: Interferometry from two RSLCs (ISCE3)
1. Choose reference and secondary RSLC: same track, frame, frequency, polarization, bandwidth
2. Get a DEM covering the scene (Copernicus GLO-30)
3. Write an ISCE3 InSAR run config (YAML) with inputs, DEM, multilook, filter, unwrapper, and ionosphere options
4. Run `python -m nisar.workflows.insar insar_runconfig.yaml`
5. The workflow runs: geo2rdr → coarse resample → dense offsets → rubbersheeting → fine resample → crossmul (interferogram + coherence) → filter → unwrap → ionosphere (split-spectrum) → geocode
6. Outputs: your own RIFG, RUNW, ROFF, and GUNW, GOFF

### Path D: Make your own GSLC or GCOV
1. Write a run config (input RSLC, DEM, output EPSG, posting, polarizations)
2. GSLC: `python -m nisar.workflows.gslc gslc_runconfig.yaml`
3. GCOV: `python -m nisar.workflows.gcov gcov_runconfig.yaml`


## RIFG: Wrapped interferogram (radar geometry)

**What it is:** the complex interferogram from two RSLCs, already coregistered, flattened, and multilooked, with phase still wrapped to (−π, π]. Includes coherence.

**Key layers**
- `/science/LSAR/RIFG/swaths/frequencyA/interferogram/HH/wrappedInterferogram` (complex)
- `.../interferogram/HH/coherenceMagnitude`
- `.../pixelOffsets/HH/` (alongTrackOffset, slantRangeOffset, correlationSurfacePeak)
- `metadata/geolocationGrid`

**When to use:** to look at raw fringes, to apply your own filter and unwrapper, or to estimate ionosphere yourself.

### Steps
1. Read the complex interferogram and coherence
2. Get wrapped phase: φ = angle(ifg)
3. Mask pixels with low coherence (for example below 0.3) and fill values
4. Apply extra multilooking if you want more noise reduction (average the complex values, not the phase)
5. Filter phase (Goldstein, α about 0.5) to reduce noise before unwrapping
6. Unwrap: SNAPHU (`snaphu-py`), using coherence as weights. Mark connected components
7. Check unwrapping: look for jumps of 2π across component boundaries; use phase closure if you have triplets
8. Correct ionosphere: split-spectrum (needs sub-band processing from RSLC) or use the RUNW ionosphere layer from the same pair
9. Set a reference point on stable ground and subtract its phase
10. Convert to LOS displacement: d = −(λ / 4π) × φ, with λ ≈ 0.24 m (one fringe ≈ 12 cm)
11. Geocode (see below)


## RUNW: Unwrapped interferogram (radar geometry)

**What it is:** the RIFG pair after filtering and unwrapping, still in radar coordinates. It includes connected components and usually an ionosphere phase screen.

**Key layers**
- `/science/LSAR/RUNW/swaths/frequencyA/interferogram/HH/unwrappedPhase`
- `.../coherenceMagnitude`, `.../connectedComponents`
- `.../ionospherePhaseScreen`, `.../ionospherePhaseScreenUncertainty`
- `.../pixelOffsets/HH/` (along-track and slant-range offsets)
- `metadata/geolocationGrid` (incidence angle, LOS vectors, coordinates)

**When to use:** deformation analysis in radar geometry, stacks for time series in radar coordinates, or when you want to control geocoding yourself.

### Steps
1. Read unwrapped phase, coherence, connected components, and ionosphere screen
2. Mask pixels with connected component = 0 (not reliably unwrapped) and low coherence
3. Pick a reference point on stable ground **inside the largest connected component**; subtract its phase. Components unwrapped separately can have different 2π offsets, so treat them separately or bridge them
4. Subtract the ionosphere phase screen (check sign convention in attributes)
5. Correct troposphere (ERA5 via PyAPS, GACOS, or a phase-elevation fit)
6. Remove residual ramps if present (plane or quadratic fit over stable areas)
7. Convert to LOS displacement: d = −(λ / 4π) × φ
8. Optional: project LOS to vertical, assuming vertical motion only: d_U = d_LOS / cos(θ)
9. Geocode with the geolocation grid or ISCE3
10. For time series: stack many RUNWs and load into MintPy with the geometry layers (radar-coordinate mode), then invert as in the GUNW workflow below


## ROFF: Pixel offsets (radar geometry)

**What it is:** sub-pixel image matching (amplitude cross-correlation) between two RSLCs. It measures large displacements in both along-track (azimuth) and slant-range directions, where phase decorrelates or wraps too many times: fast glaciers, large earthquakes, big landslides.

**Key layers**
- `/science/LSAR/ROFF/swaths/frequencyA/pixelOffsets/HH/layer1/` (and layer2, layer3, computed with different window sizes)
  - `alongTrackOffset`, `slantRangeOffset`
  - `correlationSurfacePeak`, `snr`
  - `alongTrackOffsetVariance`, `slantRangeOffsetVariance`, `crossOffsetVariance`
- `metadata/geolocationGrid`

### Steps
1. Choose the layer: small windows give finer detail but more noise; large windows give smoother, more robust results
2. Read both offset components and quality layers
3. Check units (meters or pixels) in attributes. If in pixels, multiply by pixel spacing (slant range spacing and azimuth spacing = ground velocity × azimuth time interval)
4. Mask by quality: low SNR, low correlation peak, high variance
5. Remove outliers: median filter, compare each value with its neighborhood
6. Remove bias: subtract the median offset over stable ground (rock outcrops, ice-free areas)
7. Convert slant-range offset to ground range: d_ground = d_slant / sin(θ)
8. Convert to velocity: v = offset ÷ Δt (days) × 365 for m/yr
9. Combine with the other orbit direction (ascending + descending) for 2D horizontal motion; assume surface-parallel flow for 3D
10. Geocode
11. Validate with GPS or ITS_LIVE (for glaciers)


## Geocoding Level 1 products

**Option 1: ISCE3 (most accurate)**
- Use `isce3.geocode` (or the NISAR workflows) with the product's orbit, Doppler, DEM, and output grid
- Handles terrain height, so pixels land in the right place in mountains

**Option 2: Geolocation grid interpolation (quick approach)**
1. Read `coordinateX` and `coordinateY` cubes from `metadata/geolocationGrid` (indexed by height, azimuth time, slant range)
2. Read a DEM and get the height for each radar pixel (iterate or use mean terrain height for flat areas)
3. Interpolate the cube at each pixel's azimuth time, slant range, and height to get map x, y
4. Resample the data onto a regular map grid (`scipy.interpolate.griddata` or `pyresample`)
5. Assign the CRS from the cube's EPSG attribute

Option 2 is fine for flat areas and quick looks. Use ISCE3 for quantitative work in hilly terrain.


## Level 2 products (geocoded)

Level 2 files store data on a `grids` group with `xCoordinates`, `yCoordinates`, and a `projection` (EPSG). No geocoding is needed. Metadata in `radarGrid` holds 3D cubes of incidence angle, LOS vectors, and slant range on the same map grid.

### Reading any Level 2 layer

```python
import h5py, xarray as xr, rioxarray

def read_l2(path, product, layer_path):
    with h5py.File(path, "r") as f:
        g = f"/science/LSAR/{product}/grids/frequencyA"
        data = f[f"{g}/{layer_path}"][:]
        x = f[f"{g}/xCoordinates"][:]
        y = f[f"{g}/yCoordinates"][:]
        epsg = int(f[f"{g}/projection"][()])
    da = xr.DataArray(data, coords={"y": y, "x": x}, dims=("y", "x"))
    return da.rio.write_crs(f"EPSG:{epsg}")


## GSLC: Geocoded Single Look Complex

**What it is:** the RSLC resampled onto a map grid, with complex values kept, and the phase flattened against the DEM. All GSLCs from the same track and frame share the same grid, so interferograms are a simple pixel-by-pixel multiplication.

**Key layers**
- `/science/LSAR/GSLC/grids/frequencyA/HH` (complex), plus other polarizations
- `xCoordinates`, `yCoordinates`, `projection`, `mask`
- `metadata/radarGrid/` cubes

### Path A: Interferogram
1. Choose reference and secondary GSLC: same track, frame, frequency, polarization, bandwidth
2. Confirm the x and y coordinates match exactly
3. Crop both to the AOI with the same window
4. Form the interferogram: ifg = ref × conj(sec)
5. Check metadata to confirm topographic phase was removed; if not, simulate and remove it with the DEM
6. Estimate coherence with a moving window (for example 7×7)
7. Multilook by averaging complex values
8. Filter phase (Goldstein)
9. Mask low coherence and mask layer
10. Unwrap with SNAPHU
11. Correct ionosphere (use the matching GUNW layer, or split-spectrum)
12. Reference to a stable point
13. Convert to LOS displacement
14. Export as GeoTIFF (already in map coordinates)

### Path B: Coherence time series
1. Form consecutive pairs (12-day or 24-day)
2. Compute coherence for each pair
3. Stack into a time series
4. Use for tillage and harvest detection, forest change, urban damage, flood detection in built-up areas

### Path C: Amplitude
1. Power = |GSLC|²
2. Multilook and filter
3. Use for quick-look backscatter (GCOV is better for terrain-corrected γ⁰)


## GCOV: Geocoded polarimetric covariance (γ⁰)

**What it is:** the terrain-corrected, multilooked, geocoded covariance matrix. Diagonal terms are backscatter γ⁰ in linear power. Off-diagonal terms (complex) are included for quad-pol or when requested. This is the main product for land cover, crops, biomass, and flooding.

**Key layers**
- Diagonals: `HHHH`, `HVHV`, `VHVH`, `VVVV`
- Off-diagonals: `HHHV`, `HHVV`, `HVVV`, etc.
- `mask`, `numberOfLooks`, `rtcGammaToSigmaFactor` (names may vary)

### Steps
1. Read the diagonal terms (linear γ⁰)
2. Apply the mask (layover, shadow, invalid) and fill values
3. Check `numberOfLooks`; mask or downweight low-look pixels
4. Optional: convert to σ⁰ by multiplying with `rtcGammaToSigmaFactor`
5. Optional light speckle filter, or a multitemporal filter for stacks
6. Compute indices in linear units
   - HV/HH, HH/VV
   - Quad-pol RVI = 8·HV / (HH + VV + 2·HV)
   - Dual-pol RVI = 4·HV / (HH + HV)
7. For quad-pol: build C3 from diagonal and off-diagonal terms, then run Freeman-Durden, Yamaguchi, or H-A-α decomposition
8. Convert to dB at the end: 10 × log10(γ⁰)
9. Clip and export

### Time series (crops, biomass, flooding)
1. Collect all GCOV for one track, frame, and orbit direction
2. Align to a common grid (`rio.reproject_match` if frames differ)
3. Stack in `xarray` with a time dimension
4. Multitemporal smoothing (Savitzky-Golay or rolling median)
5. Extract features (mean, percentiles, seasonal amplitude, date of peak HV)
6. Model or classify (Random Forest, XGBoost, LSTM)
7. Validate with field data, GEDI (biomass), or optical imagery


## GUNW: Geocoded unwrapped interferogram

**What it is:** the geocoded version of RUNW, with unwrapped phase, coherence, connected components, ionosphere phase screen, and pixel offsets on a map grid. It is the main input for deformation time series.

**Key layers**
- `/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/unwrappedPhase`
- `.../coherenceMagnitude`, `.../connectedComponents`
- `.../ionospherePhaseScreen`, `.../ionospherePhaseScreenUncertainty`
- `.../pixelOffsets/HH/` (along-track, slant-range)
- `metadata/radarGrid/` (incidence angle, LOS unit vectors)

### Single-pair steps
1. Read unwrapped phase, coherence, connected components, ionosphere screen
2. Mask connected component = 0 and low coherence
3. Reference to a stable point in the main connected component
4. Subtract the ionosphere phase screen
5. Correct troposphere (ERA5, GACOS, or phase-elevation fit)
6. Remove residual ramps
7. Convert to LOS displacement: d = −(λ / 4π) × φ
8. Export (already geocoded)

### Time series steps (MintPy)
1. Download all GUNW pairs for one track, frame, and orbit direction
2. Plot the baseline network and drop pairs with poor coherence or known issues
3. Set up a MintPy template with NISAR as the processor
4. Load the stack (`smallbaselineApp.py nisar_template.txt`)
5. Choose the reference point (stable, high coherence, ideally GNSS)
6. Modify the network by coherence thresholds
7. Correct unwrapping errors (bridging or phase closure, using connected components)
8. Apply the ionosphere correction
9. Invert the network to get displacement per date
10. Correct troposphere and topographic residual
11. Estimate velocity (linear + seasonal terms)
12. Mask by temporal coherence (for example above 0.7)
13. Validate with GNSS

### Ascending + descending decomposition
1. Run time series for both directions
2. Read LOS unit vectors from `metadata/radarGrid` for each
3. Resample both velocity maps to the same grid
4. Solve per pixel for east and vertical velocity:
   v_asc = e_asc·v_E + u_asc·v_U
   v_desc = e_desc·v_E + u_desc·v_U


## GOFF: Geocoded pixel offsets

**What it is:** the geocoded version of ROFF. Along-track and slant-range offsets with quality layers on a map grid, usually in several layers with different window sizes.

**Key layers**
- `/science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1` (layer2, layer3)
  - `alongTrackOffset`, `slantRangeOffset`
  - `correlationSurfacePeak`, `snr`, variance layers
- `metadata/radarGrid/` (incidence angle, LOS vectors, heading)

### Steps
1. Pick the layer (window size) suited to the feature size and noise
2. Read offsets and quality layers; check units
3. Mask by SNR, correlation peak, and variance
4. Remove outliers with a median filter
5. Remove bias over stable ground
6. Convert slant-range offset to ground range: ÷ sin(θ), using incidence angle from `radarGrid`
7. Rotate along-track and ground-range offsets to east and north using the satellite heading
8. Convert to velocity: offset ÷ Δt × 365 (m/yr)
9. Combine ascending and descending for better 2D velocity; assume surface-parallel flow for 3D
10. Fill small gaps by interpolation
11. Build time series for glacier speed-up, surge, or earthquake rupture mapping
12. Validate with ITS_LIVE, GPS, or optical feature tracking


## Summary: which product and what to do

| Product | Geometry | Main processing steps | Typical output |
|---|---|---|---|
| RSLC | Radar | Calibrate, multilook, polarimetry, or run ISCE3 InSAR/GSLC/GCOV | Custom backscatter, custom interferograms |
| RIFG | Radar | Filter, unwrap, ionosphere, reference, geocode | Your own unwrapped interferogram |
| RUNW | Radar | Mask components, reference, iono/tropo correction, LOS, geocode | Displacement map, radar-coordinate stacks |
| ROFF | Radar | Pick layer, mask, debias, ground range, velocity, geocode | Large-motion maps, glacier velocity |
| GSLC | Map | Multiply pairs, coherence, filter, unwrap | Custom interferograms, coherence time series |
| GCOV | Map | Mask, indices, decomposition, time series features | Land cover, crops, biomass, floods |
| GUNW | Map | Mask, reference, corrections, MintPy inversion | Deformation velocity and time series |
| GOFF | Map | Mask, debias, rotate to E/N, velocity | Ice velocity, earthquake offsets |


## NISAR Processing with GeoLattice


## Summary, exercises and references


### Open-source processing software

**Interferogram generation**
- **ISCE2 / ISCE3** (NASA JPL): main InSAR engine for Sentinel-1 (`topsStack`), ALOS-2, and NISAR (ISCE3 is NISAR's official processor). [isce2](https://github.com/isce-framework/isce2), [isce3](https://github.com/isce-framework/isce3)
- **ESA SNAP**: GUI and graph-based processing for Sentinel-1 interferograms, coherence, and export to SNAPHU and StaMPS. [step.esa.int](https://step.esa.int)
- **GMTSAR**: command-line InSAR built on GMT, good for Sentinel-1 and ALOS. [github.com/gmtsar/gmtsar](https://github.com/gmtsar/gmtsar)
- **PyGMTSAR**: Python version of GMTSAR, runs in Jupyter and Colab. [github.com/AlexeyPechnikov/pygmtsar](https://github.com/AlexeyPechnikov/pygmtsar)

**Phase unwrapping**
- **SNAPHU**: the standard statistical-cost unwrapper. Python wrapper: [snaphu-py](https://github.com/isce-framework/snaphu-py)

**Time series analysis**
- **MintPy**: small baseline time series, supports ISCE, HyP3, ARIA, GMTSAR, SNAP, and NISAR inputs. [github.com/insarlab/MintPy](https://github.com/insarlab/MintPy)
- **StaMPS**: persistent scatterer (PS) and SBAS, commonly paired with SNAP. [github.com/dbekaert/StaMPS](https://github.com/dbekaert/StaMPS)
- **LiCSBAS**: SBAS time series built for COMET LiCSAR products. [github.com/yumorishita/LiCSBAS](https://github.com/yumorishita/LiCSBAS)
- **Dolphin**: phase linking for distributed scatterers, used in NASA OPERA DISP products. [github.com/isce-framework/dolphin](https://github.com/isce-framework/dolphin)
- **FRInGE**: full-resolution phase linking. [github.com/isce-framework/fringe](https://github.com/isce-framework/fringe)

**Atmospheric correction**
- **PyAPS**: ERA5 tropospheric delay (built into MintPy)
- **RAiDER**: tropospheric delay from weather models. [github.com/dbekaert/RAiDER](https://github.com/dbekaert/RAiDER)
- **GACOS**: tropospheric correction maps on request. [gacos.net](http://www.gacos.net)

**Utilities**
- **ARIA-tools**: download, stitch, and prepare ARIA Sentinel-1 GUNW products for MintPy. [github.com/aria-tools/ARIA-tools](https://github.com/aria-tools/ARIA-tools)
- **asf_search**, **hyp3_sdk**, **earthaccess**, **s1-reader**, **burst2safe**: search, order, and read data in Python

### Cloud and on-demand processing

- **ASF HyP3**: free on-demand Sentinel-1 InSAR (GAMMA and ISCE burst-based), RTC, and autoRIFT offsets. [hyp3-docs.asf.alaska.edu](https://hyp3-docs.asf.alaska.edu)
- **ASF Vertex**: web search, baseline tool, and SBAS stack builder. [search.asf.alaska.edu](https://search.asf.alaska.edu)
- **ASF OpenSARLab**: hosted JupyterHub with InSAR notebooks (mainly for courses and partners)
- **COMET LiCSAR portal**: ready-made Sentinel-1 interferograms over tectonic and volcanic areas. [comet.nerc.ac.uk/comet-lics-portal](https://comet.nerc.ac.uk/comet-lics-portal/)
- **ESA Geohazards Exploitation Platform (GEP)**: hosted InSAR services (SBAS, P-SBAS, SNAPPING)

### Ready-made InSAR products

- **NASA OPERA**: CSLC-S1 (coregistered SLCs) and DISP-S1 (displacement time series) over North America
- **ARIA Sentinel-1 GUNW**: standard unwrapped interferograms over selected areas (ASF DAAC)
- **NISAR GUNW, GOFF, GSLC**: L-band products from ASF DAAC
- **European Ground Motion Service (EGMS)**: Europe-wide PS/SBAS ground motion. [egms.land.copernicus.eu](https://egms.land.copernicus.eu)
- **COMET LiCSAR**: global tectonic and volcanic interferograms
- **ITS_LIVE**: glacier velocity from offsets (optical and SAR). [its-live.jpl.nasa.gov](https://its-live.jpl.nasa.gov)

### Training and tutorials

- **NASA ARSET**: free SAR and InSAR webinars with recordings and exercises
- **EarthScope (formerly UNAVCO) InSAR short courses**: ISCE and MintPy hands-on training
- **ASF SAR training and data recipes**: step-by-step InSAR guides and Jupyter notebooks
- **EO College "Echoes in Space"**: free SAR and InSAR online course (ESA/DLR)
- **ESA SNAP tutorials**: Sentinel-1 interferometry and StaMPS export guides on the STEP site
- **MintPy tutorials**: example notebooks in the MintPy repository
- **NISAR Science Users' Handbook** and the **ASF NISAR Data User Guide**

### Books and reference texts

- Hanssen, R. (2001). *Radar Interferometry: Data Interpretation and Error Analysis*. Kluwer
- Ferretti, A. (2014). *Satellite InSAR Data: Reservoir Monitoring from Space*. EAGE
- Flores-Anderson, A. et al. (2019). *The SAR Handbook*. SERVIR / NASA (free PDF)
- Bamler, R. & Hartl, P. (1998). "Synthetic aperture radar interferometry." *Inverse Problems* 14
- Rosen, P. et al. (2000). "Synthetic aperture radar interferometry." *Proceedings of the IEEE* 88(3)
- Yunjun, Z., Fattahi, H. & Amelung, F. (2019). "Small baseline InSAR time series analysis: Unwrapping error correction and noise reduction." *Computers & Geosciences* (the MintPy paper)

### Commercial software

- **GAMMA** (full InSAR and PS suite)
- **ENVI SARscape**
- **SARPROZ** (PS and SBAS, MATLAB-based)
- **SkyGeo, TRE Altamira, 3vG**: commercial InSAR monitoring services

### Communities and support

- **STEP Forum** (SNAP and Sentinel-1 questions): [forum.step.esa.int](https://forum.step.esa.int)
- **NASA Earthdata Forum** (ASF, HyP3, OPERA, NISAR)
- **GitHub Discussions and Issues** for MintPy, ISCE, and Dolphin
- **FRINGE workshop** (ESA): main InSAR science conference

### Suggested starting path

1. Order Sentinel-1 interferograms from **ASF HyP3** (no local processing)
2. Run time series in **MintPy**
3. Move to **ISCE2 topsStack** or **SNAP + StaMPS** when you need full control
4. Use **OPERA DISP-S1** for North America and **EGMS** for Europe when you only need results
5. Add **NISAR GUNW** for vegetated areas where C-band loses coherence

I can put together a HyP3 + MintPy notebook as a starter example for your Remote Sensing with Python tutorial.


**Installation.** This notebook needs installation option 1 (vector, raster, CRS and visualization): `pip install "geolattice[core]"`. See "Installation options" in the GeoLattice README.